In [ ]:
# TrustCart – RNN (Customer Path Anomaly Detection)

In [ ]:
import json
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss, classification_report, confusion_matrix,
                             precision_recall_curve, roc_auc_score, roc_curve)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers

pd.set_option("display.max_columns", 50)
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

In [ ]:
def find_dataset_dir(filename="events.csv", root="/kaggle/input"):
    """Kaggle mounts datasets at different paths in interactive and batch sessions: search for it."""
    for dirpath, _, files in os.walk(root):
        if filename in files:
            return dirpath
    raise FileNotFoundError(f"{filename} not found under {root}: attach the Retailrocket dataset")


DATA_DIR = find_dataset_dir()
OUT_DIR = "/kaggle/working"          # the only folder where Kaggle lets you write

EVENTS_PATH = os.path.join(DATA_DIR, "events.csv")
PROPS_PATHS = [os.path.join(DATA_DIR, "item_properties_part1.csv"),
               os.path.join(DATA_DIR, "item_properties_part2.csv")]
TREE_PATH = os.path.join(DATA_DIR, "category_tree.csv")

for p in [EVENTS_PATH, *PROPS_PATHS, TREE_PATH]:
    print(("OK      " if os.path.exists(p) else "MISSING ") + p,
          f"({os.path.getsize(p)/1e6:.1f} MB)" if os.path.exists(p) else "")

In [ ]:
CONFIG = {
    "session_gap_min": 30,     # inactivity that ends a session
    "min_session_len": 3,      # shorter sessions are dropped
    "max_session_len": 50,     # longer ones are cut (and used for padding size)
    "anomaly_ratio": 0.10,     # share of anomalous sessions we will inject
    "test_size": 0.15,
    "val_size": 0.15,
}
CONFIG


In [ ]:
events = pd.read_csv(EVENTS_PATH)

print("Shape:", events.shape)
print("Memory: %.1f MB" % (events.memory_usage(deep=True).sum() / 1e6))
display(events.head(10))
events.info()

In [ ]:
counts = events["event"].value_counts()
print(pd.DataFrame({"count": counts, "percent": (counts / len(events) * 100).round(2)}))

In [ ]:
events["datetime"] = pd.to_datetime(events["timestamp"], unit="ms")
print("From:", events["datetime"].min())
print("To:  ", events["datetime"].max())
print("Days:", (events["datetime"].max() - events["datetime"].min()).days)

daily = events.set_index("datetime").resample("D").size()
daily.plot(figsize=(12, 3), title="Events per day")
plt.ylabel("events"); plt.show()

In [ ]:
n_visitors = events["visitorid"].nunique()
per_visitor = events.groupby("visitorid").size()

print("Unique visitors:", n_visitors)
print("Unique items:   ", events["itemid"].nunique())
print("\nEvents per visitor:")
print(per_visitor.describe(percentiles=[.5, .75, .9, .99]).round(1))

has = events.groupby("visitorid")["event"].agg(set)
print("\nVisitors with at least one addtocart:  ", has.apply(lambda s: "addtocart" in s).sum())
print("Visitors with at least one transaction:", has.apply(lambda s: "transaction" in s).sum())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
per_visitor.clip(upper=50).hist(bins=50, ax=ax[0])
ax[0].set_title("Events per visitor (clipped at 50)"); ax[0].set_yscale("log")

events["hour"] = events["datetime"].dt.hour
events["hour"].value_counts().sort_index().plot(kind="bar", ax=ax[1], title="Events by hour (UTC)")
plt.tight_layout(); plt.show()

In [ ]:
events = events.drop(columns=["hour"])          # helper column from Phase 1, not needed anymore

n_before = len(events)
events = events.drop_duplicates(subset=["timestamp", "visitorid", "event", "itemid"])
print(f"Duplicates removed: {n_before - len(events):,}  ({(n_before - len(events)) / n_before:.3%})")

In [ ]:
EVENT_ORDER = {"view": 0, "addtocart": 1, "transaction": 2}
events["event_rank"] = events["event"].map(EVENT_ORDER)

events = (events
          .sort_values(["visitorid", "timestamp", "event_rank"], kind="mergesort")
          .reset_index(drop=True))

# Check: within each visitor, time must never go backwards
same_visitor = events["visitorid"].eq(events["visitorid"].shift())
backwards = same_visitor & (events["timestamp"].diff() < 0)
print("Out-of-order events:", int(backwards.sum()))
display(events.head(10))

In [ ]:
per_visitor = events.groupby("visitorid").size()
threshold = per_visitor.quantile(0.999)

# Visitors who never picked or bought anything
active = events.loc[events["event"].isin(["addtocart", "transaction"]), "visitorid"].unique()

heavy = per_visitor[per_visitor > threshold].index
bots = heavy.difference(active)          # extreme activity AND zero purchase intent

bot_events = events["visitorid"].isin(bots)
print(f"Threshold (99.9th percentile): {threshold:.0f} events")
print(f"Heavy visitors:       {len(heavy):,}")
print(f"  of which bots kept out (no addtocart/transaction): {len(bots):,}")
print(f"  of which real shoppers kept in:                    {len(heavy) - len(bots):,}")
print(f"Events removed: {bot_events.sum():,}  ({bot_events.mean():.2%} of events)")

events = events[~bot_events].reset_index(drop=True)
print("Remaining events:", f"{len(events):,}")

In [ ]:
events = events.drop(columns=["transactionid", "event_rank"])
events["event"] = events["event"].astype("category")
events["visitorid"] = events["visitorid"].astype("int32")
events["itemid"] = events["itemid"].astype("int32")

print("Memory: %.1f MB" % (events.memory_usage(deep=True).sum() / 1e6))
events.info()

In [ ]:
CLEAN_PATH = os.path.join(OUT_DIR, "events_clean.parquet")
events.to_parquet(CLEAN_PATH, index=False)
print("Saved:", CLEAN_PATH, f"({os.path.getsize(CLEAN_PATH) / 1e6:.1f} MB)")

In [ ]:
CLEAN_PATH = os.path.join(OUT_DIR, "events_clean.parquet")
events = pd.read_parquet(CLEAN_PATH)
print(f"{len(events):,} events loaded")

In [ ]:
parts = []
for path in PROPS_PATHS:
    for chunk in pd.read_csv(path, chunksize=2_000_000,
                             dtype={"property": str, "value": str}):
        parts.append(chunk.loc[chunk["property"] == "categoryid",
                               ["timestamp", "itemid", "value"]])
props = pd.concat(parts, ignore_index=True)
props["categoryid"] = props["value"].astype("int32")
props = props.drop(columns="value")

print(f"categoryid rows: {len(props):,}")
print(f"Items with a category: {props['itemid'].nunique():,}")

In [ ]:
moved = props.groupby("itemid")["categoryid"].nunique()
print(f"Items whose category changed over time: {(moved > 1).sum():,}  ({(moved > 1).mean():.2%})")

item_cat = (props.sort_values("timestamp")
                 .drop_duplicates("itemid", keep="last")[["itemid", "categoryid"]])
print(f"Final item → category table: {len(item_cat):,} items")

In [ ]:
tree = pd.read_csv(TREE_PATH)
parent = tree.dropna().astype(int).set_index("categoryid")["parentid"].to_dict()

def root_and_depth(c):
    depth, seen = 0, set()
    while c in parent and c not in seen:      # 'seen' protects against cycles
        seen.add(c)
        c = parent[c]
        depth += 1
    return c, depth

roots = {c: root_and_depth(c) for c in set(tree["categoryid"]) | set(item_cat["categoryid"])}
item_cat["root"] = item_cat["categoryid"].map(lambda c: roots[c][0])

depths = pd.Series({c: d for c, (_, d) in roots.items()})
print("Categories in the tree:", len(tree))
print("Root categories (aisles):", item_cat["root"].nunique())
print("Tree depth distribution:\n", depths.value_counts().sort_index())


In [ ]:
events = events.merge(item_cat[["itemid", "categoryid", "root"]], on="itemid", how="left")

coverage = events["root"].notna().mean()
print(f"Events with a known aisle: {coverage:.1%}")

# Aisle IDs: 1..K for real aisles, 0 = unknown
root_ids = sorted(events["root"].dropna().unique().astype(int))
AISLE_MAP = {r: i + 1 for i, r in enumerate(root_ids)}
events["aisle"] = events["root"].map(AISLE_MAP).fillna(0).astype("int16")
events["categoryid"] = events["categoryid"].fillna(-1).astype("int32")
events = events.drop(columns="root")

print("Number of aisles (excluding unknown):", len(AISLE_MAP))
events["aisle"].value_counts().sort_index().plot(kind="bar", figsize=(12, 3),
                                                  title="Events per aisle (0 = unknown)")
plt.show()

In [ ]:
AISLES_PATH = os.path.join(OUT_DIR, "events_aisles.parquet")
events.to_parquet(AISLES_PATH, index=False)
print("Saved:", AISLES_PATH, f"({os.path.getsize(AISLES_PATH) / 1e6:.1f} MB)")
del props, parts

In [ ]:
AISLES_PATH = os.path.join(OUT_DIR, "events_aisles.parquet")
events = pd.read_parquet(AISLES_PATH)

EVENT_ORDER = {"view": 0, "addtocart": 1, "transaction": 2}
events["event_rank"] = events["event"].astype(str).map(EVENT_ORDER)
events = (events.sort_values(["visitorid", "timestamp", "event_rank"], kind="mergesort")
                .drop(columns="event_rank")
                .reset_index(drop=True))
print(f"{len(events):,} events loaded")

In [ ]:
gap_ms = CONFIG["session_gap_min"] * 60 * 1000

same_visitor = events["visitorid"].eq(events["visitorid"].shift())
dt_ms = events["timestamp"].diff()
new_session = ~same_visitor | (dt_ms > gap_ms)

events["session_id"] = new_session.cumsum().astype("int32")
events["dt_sec"] = np.where(new_session, 0, dt_ms / 1000).astype("float32")

print(f"Sessions created: {events['session_id'].nunique():,}")
display(events.head(12))

In [ ]:
ev = events["event"].astype(str)
sess = (events.assign(is_view=ev.eq("view"), is_add=ev.eq("addtocart"), is_tx=ev.eq("transaction"))
              .groupby("session_id")
              .agg(visitorid=("visitorid", "first"),
                   n_events=("timestamp", "size"),
                   n_view=("is_view", "sum"),
                   n_add=("is_add", "sum"),
                   n_tx=("is_tx", "sum"),
                   start=("timestamp", "min"),
                   end=("timestamp", "max")))
sess["duration_min"] = (sess["end"] - sess["start"]) / 60000

print(f"Total sessions: {len(sess):,}")
print(f"Sessions with ≥1 addtocart: {(sess['n_add'] > 0).sum():,}  ({(sess['n_add'] > 0).mean():.2%})")
print("\nSession length (events):")
print(sess["n_events"].describe(percentiles=[.5, .9, .99]).round(1))

In [ ]:
keep = (sess["n_add"] >= 1) & (sess["n_events"] >= CONFIG["min_session_len"])
kept = sess[keep]

print(f"Sessions kept: {len(kept):,}  ({keep.mean():.2%} of all sessions)")
print(f"  ending with a purchase:      {(kept['n_tx'] > 0).sum():,}")
print(f"  picked but nothing paid:     {(kept['n_tx'] == 0).sum():,}")
print(f"  longer than {CONFIG['max_session_len']} events (will be cut): "
      f"{(kept['n_events'] > CONFIG['max_session_len']).sum():,}")

events = events[events["session_id"].isin(kept.index)].reset_index(drop=True)

In [ ]:
MAX_LEN = CONFIG["max_session_len"]

step_from_end = events.groupby("session_id").cumcount(ascending=False)
events = events[step_from_end < MAX_LEN].reset_index(drop=True)

# The first kept event of a cut session no longer has a previous event
first = ~events["session_id"].eq(events["session_id"].shift())
events.loc[first, "dt_sec"] = 0.0
events["step"] = events.groupby("session_id").cumcount().astype("int16")

lengths = events.groupby("session_id").size()
print(f"Events kept: {len(events):,}")
print(f"Max session length now: {lengths.max()}")

In [ ]:
lengths = events.groupby("session_id").size()
dur = sess.loc[lengths.index, "duration_min"]

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
lengths.hist(bins=50, ax=ax[0]); ax[0].set_title("Events per kept session")
dur.clip(upper=120).hist(bins=60, ax=ax[1]); ax[1].set_title("Session duration (min, clipped at 120)")
plt.tight_layout(); plt.show()

# One real example
sid = sess.loc[lengths.index].query("n_tx > 0").index[0]
display(events[events["session_id"] == sid][["datetime", "event", "itemid", "aisle", "dt_sec"]])

In [ ]:
SESS_PATH = os.path.join(OUT_DIR, "sessions_events.parquet")
events.to_parquet(SESS_PATH, index=False)
print("Saved:", SESS_PATH, f"({os.path.getsize(SESS_PATH) / 1e6:.1f} MB)",
      f"| {events['session_id'].nunique():,} sessions")

In [ ]:
SESS_PATH = os.path.join(OUT_DIR, "sessions_events.parquet")
events = pd.read_parquet(SESS_PATH)
print(f"{len(events):,} events | {events['session_id'].nunique():,} sessions")

## In-store mapping (modeling assumptions)

| Online event | In-store meaning |
|---|---|
| `view` | LOOK – customer is in an aisle, looking at a product |
| `addtocart` | PICK – customer puts the product in the basket |
| `transaction` | PAY – the product goes through checkout |
| item picked but never paid in the session | PUT BACK – customer changed their mind (normal) |
| `dt_sec` | time spent since the previous action |
| item root category | store aisle (23 active aisles + 0 = unknown) |
| price | synthetic, per item, consistent across sessions (real prices are hashed) |


In [ ]:
ACTION_MAP = {"view": "LOOK", "addtocart": "PICK", "transaction": "PAY"}
events["action"] = events["event"].astype(str).map(ACTION_MAP).astype("category")
print(events["action"].value_counts())

In [ ]:
rng = np.random.default_rng(SEED)

TIER_PRICE = {"cheap": 3.0, "medium": 15.0, "expensive": 80.0}   # typical price per tier (EUR)
aisles = sorted(events["aisle"].unique())
aisle_tier = {a: rng.choice(list(TIER_PRICE), p=[0.4, 0.4, 0.2]) for a in aisles}
aisle_tier[0] = "medium"                                            # unknown aisle = neutral

items = events[["itemid", "aisle"]].drop_duplicates("itemid").reset_index(drop=True)
base = items["aisle"].map(aisle_tier).map(TIER_PRICE)
items["price"] = (base * rng.lognormal(mean=0.0, sigma=0.5, size=len(items))).round(2)

summary = (items.groupby("aisle")
                .agg(tier=("aisle", lambda s: aisle_tier[s.iloc[0]]),
                     n_items=("itemid", "size"),
                     median_price=("price", "median"))
                .round(2))
display(summary)

In [ ]:
events = events.merge(items[["itemid", "price"]], on="itemid", how="left")
assert events["price"].notna().all(), "Some events have no price"

events["price"].clip(upper=300).hist(bins=60, figsize=(8, 3))
plt.title("Item price distribution (clipped at 300 EUR)"); plt.show()
print(events.groupby("action", observed=True)["price"].describe().round(2))

In [ ]:
dt = events.loc[events["step"] > 0, "dt_sec"]
pick = events["action"] == "PICK"

PROFILE = {
    "dt_p50":  float(dt.quantile(0.50)),
    "dt_p90":  float(dt.quantile(0.90)),
    "dt_p99":  float(dt.quantile(0.99)),
    "pick_price_p50": float(events.loc[pick, "price"].quantile(0.50)),
    "pick_price_p90": float(events.loc[pick, "price"].quantile(0.90)),
    "picks_per_session_p90": float(events[pick].groupby("session_id").size().quantile(0.90)),
}
for k, v in PROFILE.items():
    print(f"{k:24s} {v:10.1f}")

pd.Series(dt).clip(upper=1800).hist(bins=60, figsize=(8, 3))
plt.title("Time between actions (s, clipped at 30 min)"); plt.show()

In [ ]:
EVENTS_STORE_PATH = os.path.join(OUT_DIR, "sessions_store.parquet")
PRICES_PATH = os.path.join(OUT_DIR, "item_prices.csv")

events.to_parquet(EVENTS_STORE_PATH, index=False)
items.to_csv(PRICES_PATH, index=False)
print("Saved:", EVENTS_STORE_PATH)
print("Saved:", PRICES_PATH, f"({len(items):,} items)")

In [ ]:
STORE_PATH = os.path.join(OUT_DIR, "sessions_store.parquet")
events = pd.read_parquet(STORE_PATH)
events["action"] = events["action"].astype(str)
events = events.sort_values(["session_id", "step"]).reset_index(drop=True)
MAX_LEN = CONFIG["max_session_len"]

COLS = ["action", "itemid", "aisle", "price", "dt_sec"]
sessions = {sid: g[COLS].to_dict("records") for sid, g in events.groupby("session_id", sort=True)}
print(f"{len(sessions):,} normal sessions loaded")

In [ ]:
rng = np.random.default_rng(SEED)

dt_real = events.loc[events["step"] > 0, "dt_sec"].to_numpy()
picks = events[events["action"] == "PICK"]

INJ = {
    "dt_fast":         max(float(np.quantile(dt_real, 0.10)), 2.0),   # a "rapid" action
    "dt_long_lo":      float(np.quantile(dt_real, 0.99)),             # start of a suspicious stop
    "dt_long_hi":      CONFIG["session_gap_min"] * 60 - 10,           # must stay under the session cut-off
    "price_expensive": float(picks["price"].quantile(0.90)),
    "price_cheap":     float(picks["price"].quantile(0.25)),
}

ITEM_COLS = ["itemid", "aisle", "price"]
expensive_pool = picks.loc[picks["price"] > INJ["price_expensive"], ITEM_COLS].to_dict("records")
cheap_pool = picks.loc[picks["price"] <= INJ["price_cheap"], ITEM_COLS].to_dict("records")

mid_plus = picks[picks["price"] >= picks["price"].median()]
spree_pool = {int(a): g[ITEM_COLS].to_dict("records")
              for a, g in mid_plus.groupby("aisle") if len(g) >= 20}

for k, v in INJ.items():
    print(f"{k:16s} {v:8.1f}")
print(f"\nExpensive pool: {len(expensive_pool):,} picks | Cheap pool: {len(cheap_pool):,} picks")
print("Spree aisles:", sorted(spree_pool))

In [ ]:
def pick_one(pool):
    return pool[int(rng.integers(len(pool)))]

def make_event(action, item, dt):
    return {"action": action, "itemid": item["itemid"], "aisle": item["aisle"],
            "price": item["price"], "dt_sec": float(dt)}

def natural_dt():
    """A real time gap, drawn from the observed distribution."""
    return float(rng.choice(dt_real))

def drop_looks(seq, k):
    """Remove k random LOOK events so the anomaly keeps the original length."""
    looks = [i for i, e in enumerate(seq) if e["action"] == "LOOK"]
    drop = {int(i) for i in rng.choice(looks, size=min(k, len(looks)), replace=False)} if looks else set()
    return [e for i, e in enumerate(seq) if i not in drop]

def checkout_start(seq):
    """Index where the final block of PAY events (the checkout) begins; len(seq) if none."""
    i = len(seq)
    while i > 0 and seq[i - 1]["action"] == "PAY":
        i -= 1
    return i

def finalize(seq):
    seq = seq[-MAX_LEN:]
    seq[0] = {**seq[0], "dt_sec": 0.0}
    return seq

In [ ]:
def insert_pos(seq, end):
    """A random position before the final checkout that never splits a group of PAY events."""
    ok = [i for i in range(end) if seq[i]["action"] != "PAY"]
    return int(rng.choice(ok)) if ok else end

def conceal(seq):
    """Expensive item picked, then a long stop (hiding it); it is never paid."""
    seq = drop_looks(seq, 2)                                   # 2 inserted, 2 removed
    item = pick_one(expensive_pool)
    pos = insert_pos(seq, checkout_start(seq))
    seq[pos:pos] = [make_event("LOOK", item, natural_dt()),
                    make_event("PICK", item, rng.uniform(5, 60))]
    if pos + 2 < len(seq):     # the action after the pick comes after a suspicious stop
        seq[pos + 2] = {**seq[pos + 2],
                        "dt_sec": float(rng.uniform(INJ["dt_long_lo"], INJ["dt_long_hi"]))}
    return seq

def alibi(seq):
    """Expensive item kept, then a cheap item grabbed just before checkout and paid alone."""
    seq = drop_looks(seq, 4)                                   # 4 inserted, 4 removed
    exp, cheap = pick_one(expensive_pool), pick_one(cheap_pool)
    end = checkout_start(seq)
    pos = insert_pos(seq, end)
    seq[pos:pos] = [make_event("LOOK", exp, natural_dt()),
                    make_event("PICK", exp, rng.uniform(5, 60))]
    end += 2
    seq[end:end] = [make_event("PICK", cheap, rng.uniform(3, 30))]
    has_checkout = end + 1 < len(seq)
    seq.append(make_event("PAY", cheap, rng.uniform(0.1, 2) if has_checkout else rng.uniform(20, 120)))
    return seq

def spree(seq):
    """Several items grabbed within seconds in the same aisle, none of them paid."""
    n = int(rng.integers(3, 6))
    seq = drop_looks(seq, n)                                   # n inserted, n removed
    aisles = list(spree_pool)
    pool = spree_pool[aisles[int(rng.integers(len(aisles)))]]
    pos = insert_pos(seq, checkout_start(seq))
    seq[pos:pos] = [make_event("PICK", pick_one(pool),
                               natural_dt() if i == 0 else rng.uniform(1, INJ["dt_fast"]))
                    for i in range(n)]
    return seq

PATTERNS = {"conceal": conceal, "alibi": alibi, "spree": spree}

In [ ]:
normal_ids = np.array(sorted(sessions))
n_anom = int(round(len(normal_ids) * CONFIG["anomaly_ratio"]))
chosen = rng.choice(normal_ids, size=n_anom, replace=False)
names = rng.choice(list(PATTERNS), size=n_anom)

next_id = int(normal_ids.max()) + 1
anomalies = []                                  # (new_id, orig_id, pattern, sequence)
for i, (orig, name) in enumerate(zip(chosen, names)):
    seq = [dict(e) for e in sessions[orig]]     # copy: never modify the original
    seq = finalize(PATTERNS[name](seq))
    anomalies.append((next_id + i, int(orig), str(name), seq))

print(f"Anomalies created: {len(anomalies):,}")
print(pd.Series(names).value_counts())

In [ ]:
rows = []
for sid, seq in sessions.items():
    for step, e in enumerate(seq):
        rows.append({**e, "session_id": sid, "orig_id": sid, "step": step, "label": 0, "pattern": "none"})
for sid, orig, name, seq in anomalies:
    for step, e in enumerate(seq):
        rows.append({**e, "session_id": sid, "orig_id": orig, "step": step, "label": 1, "pattern": name})

data = pd.DataFrame(rows)
labels = data.groupby("session_id")["label"].first()
print(f"Sessions: {len(labels):,}  |  normal: {(labels == 0).sum():,}  |  anomalies: {(labels == 1).sum():,}"
      f"  ({labels.mean():.1%})")

In [ ]:
d = data.assign(is_pick=data["action"].eq("PICK"), is_pay=data["action"].eq("PAY"))
per_sess = d.groupby("session_id").agg(
    pattern=("pattern", "first"),
    length=("step", "size"),
    pick_share=("is_pick", "mean"),
    pay_share=("is_pay", "mean"),
    mean_price=("price", "mean"),
    total_min=("dt_sec", "sum"),
    max_dt=("dt_sec", "max"),
)
per_sess["total_min"] /= 60
display(per_sess.groupby("pattern").mean().round(2))

fig, ax = plt.subplots(figsize=(8, 3))
for name, g in per_sess.groupby("pattern"):
    g["length"].clip(upper=50).plot(kind="hist", bins=25, alpha=0.4, density=True, ax=ax, label=name)
ax.legend(); ax.set_title("Session length by pattern"); plt.show()

In [ ]:
show = ["action", "itemid", "aisle", "price", "dt_sec"]
for name in ["conceal", "alibi", "spree"]:
    sid = per_sess.query("pattern == @name").index[0]
    orig = data.loc[data["session_id"] == sid, "orig_id"].iloc[0]
    print(f"\n===== {name.upper()}  (anomaly {sid}, copied from {orig}) =====")
    print("--- original ---")
    display(data.loc[data["session_id"] == orig, show].round(1))
    print("--- anomaly ---")
    display(data.loc[data["session_id"] == sid, show].round(1))

In [ ]:
import json

assert data["label"].sum() > 0, "No anomalies in the dataset: re-run cells 36 to 41 first"

LABELED_PATH = os.path.join(OUT_DIR, "sessions_labeled.parquet")
data.to_parquet(LABELED_PATH, index=False)
with open(os.path.join(OUT_DIR, "injection_config.json"), "w") as f:
    json.dump(INJ, f, indent=2)

print("Saved:", LABELED_PATH, f"({os.path.getsize(LABELED_PATH) / 1e6:.1f} MB)")
print("Saved: injection_config.json")

## Phase 7 – Encode the sequences

Each session becomes three aligned arrays of length `MAX_LEN` (50), **right-padded** with `0`:

| Input | Shape | Content |
|---|---|---|
| `X_action` | `(N, 50)` int | 1 = LOOK, 2 = PICK, 3 = PAY, 0 = padding |
| `X_aisle` | `(N, 50)` int | aisle + 1 (1 = unknown aisle, 2…24 = real aisles), 0 = padding |
| `X_num` | `(N, 50, 2)` float | `log1p(dt_sec)`, `log1p(price)`, 0 on padding |

**Design decisions**
- **`itemid` is not a model input.** 37k hashed ids with few examples each would be memorised rather than learned, and the injected anomalies reuse a limited pool of items: an item embedding would be a leakage channel.
- **`log1p` on time and price:** both are heavy-tailed (Phase 5); the log compresses the tail so one 600 € item or a 29-minute pause does not dominate the gradients.
- **No standardisation here.** Mean/std are learnt from the *training split only*, inside the model (Keras `Normalization` layer, Phase 9). Fitting them on all sessions would leak test statistics, and keeping them inside the model guarantees training and inference use identical scaling.
- **Right padding:** the GPU (cuDNN) LSTM kernel only supports masks that are right-padded.
- **One encoding function for training and inference:** `encode_session()` is the reference implementation; the vectorised batch path is checked against it.

In [ ]:
LABELED_PATH = os.path.join(OUT_DIR, "sessions_labeled.parquet")
data = pd.read_parquet(LABELED_PATH)
data = data.sort_values(["session_id", "step"]).reset_index(drop=True)

# Contract with Phase 6: contiguous steps, one label per session, no missing values
assert (data["step"].to_numpy() == data.groupby("session_id").cumcount().to_numpy()).all(), "Non-contiguous steps"
assert data.groupby("session_id")["label"].nunique().eq(1).all(), "Mixed labels inside a session"
assert data[["action", "aisle", "price", "dt_sec"]].notna().all().all(), "Missing values"
assert data["step"].max() < CONFIG["max_session_len"], "A session is longer than max_session_len"

session_labels = data.groupby("session_id")["label"].first()
print(f"{len(data):,} events | {len(session_labels):,} sessions | anomaly rate {session_labels.mean():.1%}")

In [ ]:
MAX_LEN = CONFIG["max_session_len"]
PAD = 0

ACTION_VOCAB = {"LOOK": 1, "PICK": 2, "PAY": 3}      # 0 is reserved for padding
AISLE_OFFSET = 1                                     # aisle 0 (unknown) is a real token, so shift by 1
N_AISLES = int(data["aisle"].max()) + 1
AISLE_VOCAB_SIZE = N_AISLES + AISLE_OFFSET           # padding + unknown + real aisles
NUM_FEATURES = ["log_dt", "log_price"]


def numeric_features(dt_sec, price):
    """Stateless transforms shared by training and inference (scaling is learnt inside the model)."""
    dt_sec = np.clip(np.asarray(dt_sec, dtype="float32"), 0, None)
    price = np.clip(np.asarray(price, dtype="float32"), 0, None)
    return np.stack([np.log1p(dt_sec), np.log1p(price)], axis=-1)


def encode_session(actions, aisles, dt_sec, price, max_len=MAX_LEN):
    """Reference encoder for one ordered session -> (action_ids, aisle_ids, numeric), right-padded.

    Mirrors the training data conventions: keeps the last `max_len` events, the first kept
    event has dt = 0, and aisles outside the known range are mapped to the unknown aisle.
    """
    actions = list(actions)[-max_len:]
    aisles = np.asarray(aisles, dtype="int64")[-max_len:]
    dt_sec = np.asarray(dt_sec, dtype="float32")[-max_len:].copy()
    price = np.asarray(price, dtype="float32")[-max_len:]
    n = len(actions)
    if n == 0:
        raise ValueError("Cannot encode an empty session")
    dt_sec[0] = 0.0
    aisles = np.where((aisles >= 0) & (aisles < N_AISLES), aisles, 0)

    action_ids = np.full(max_len, PAD, dtype="int32")
    aisle_ids = np.full(max_len, PAD, dtype="int32")
    numeric = np.zeros((max_len, len(NUM_FEATURES)), dtype="float32")
    action_ids[:n] = [ACTION_VOCAB[a] for a in actions]   # unknown action -> KeyError (fail fast)
    aisle_ids[:n] = aisles + AISLE_OFFSET
    numeric[:n] = numeric_features(dt_sec, price)
    return action_ids, aisle_ids, numeric


print(f"Action vocab: {ACTION_VOCAB}  (+ padding)")
print(f"Aisle vocab size: {AISLE_VOCAB_SIZE}  (padding + unknown + {N_AISLES - 1} aisles)")

In [ ]:
# One row per session, in the same order as the tensors below
meta = (data.groupby("session_id", sort=True)
            .agg(orig_id=("orig_id", "first"), label=("label", "first"),
                 pattern=("pattern", "first"), length=("step", "size"))
            .reset_index())

row = data.groupby("session_id", sort=True).ngroup().to_numpy()   # tensor row of each event
col = data["step"].to_numpy()                                     # position inside the sequence
N = len(meta)

X_action = np.full((N, MAX_LEN), PAD, dtype="int32")
X_aisle = np.full((N, MAX_LEN), PAD, dtype="int32")
X_num = np.zeros((N, MAX_LEN, len(NUM_FEATURES)), dtype="float32")

X_action[row, col] = data["action"].map(ACTION_VOCAB).to_numpy()
X_aisle[row, col] = data["aisle"].to_numpy() + AISLE_OFFSET
X_num[row, col] = numeric_features(data["dt_sec"], data["price"])

y = meta["label"].to_numpy(dtype="int32")
lengths = meta["length"].to_numpy(dtype="int32")

print(f"X_action {X_action.shape} | X_aisle {X_aisle.shape} | X_num {X_num.shape} | y {y.shape}")
print(f"Padding share: {(X_action == PAD).mean():.1%} of all positions")

In [ ]:
mask = X_action != PAD

assert (mask.sum(axis=1) == lengths).all(), "Mask length differs from session length"
assert (np.diff(mask.astype("int8"), axis=1) <= 0).all(), "Padding is not strictly on the right"
assert (X_aisle[~mask] == PAD).all() and (X_num[~mask] == 0).all(), "Non-zero values in padding"
assert (X_aisle[mask] >= 1).all() and (X_aisle < AISLE_VOCAB_SIZE).all(), "Aisle id out of range"
assert np.isfinite(X_num).all(), "NaN or inf in numeric features"
assert (X_num[:, 0, 0] == 0).all(), "First event of a session must have dt = 0"

# The vectorised path must match the reference encoder used at inference (no train/serve skew)
check_rng = np.random.default_rng(SEED)
rows_of = data.groupby("session_id", sort=True).indices
for i in check_rng.choice(N, size=300, replace=False):
    ev = data.iloc[rows_of[meta.at[i, "session_id"]]]
    a, s, x = encode_session(ev["action"], ev["aisle"], ev["dt_sec"], ev["price"])
    assert np.array_equal(a, X_action[i]) and np.array_equal(s, X_aisle[i]), f"Mismatch on row {i}"
    assert np.allclose(x, X_num[i]), f"Numeric mismatch on row {i}"

print("All integrity checks passed (300 sessions re-encoded with encode_session).")

In [ ]:
event_label = np.broadcast_to(y[:, None], mask.shape)[mask]

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
for j, name in enumerate(NUM_FEATURES):
    values = X_num[..., j][mask]
    for lab, txt in [(0, "normal"), (1, "anomaly")]:
        ax[j].hist(values[event_label == lab], bins=60, alpha=0.5, density=True, label=txt)
    ax[j].set_title(f"{name} (real events only)")
    ax[j].legend()
plt.tight_layout(); plt.show()

i = int(np.flatnonzero(y == 1)[0])
print(f"Encoded anomaly example (session {meta.at[i, 'session_id']}, pattern {meta.at[i, 'pattern']}):")
display(pd.DataFrame({"action": X_action[i], "aisle": X_aisle[i],
                      "log_dt": X_num[i, :, 0].round(2), "log_price": X_num[i, :, 1].round(2)})
          .head(lengths[i] + 2))

In [ ]:
ENCODED_PATH = os.path.join(OUT_DIR, "sequences_encoded.npz")
ENCODERS_PATH = os.path.join(OUT_DIR, "encoders.json")

np.savez_compressed(
    ENCODED_PATH,
    X_action=X_action, X_aisle=X_aisle, X_num=X_num, y=y, lengths=lengths,
    session_id=meta["session_id"].to_numpy(), orig_id=meta["orig_id"].to_numpy(),
    pattern=meta["pattern"].to_numpy().astype(str),
)

ENCODERS = {
    "max_len": MAX_LEN,
    "pad_id": PAD,
    "action_vocab": ACTION_VOCAB,
    "aisle_offset": AISLE_OFFSET,
    "n_aisles": N_AISLES,
    "aisle_vocab_size": AISLE_VOCAB_SIZE,
    "num_features": NUM_FEATURES,
    "numeric_transform": "log1p, clipped at 0; standardisation is inside the model",
    "truncation": "keep last max_len events, first kept event gets dt = 0",
    "padding": "right",
}
with open(ENCODERS_PATH, "w") as f:
    json.dump(ENCODERS, f, indent=2)

print("Saved:", ENCODED_PATH, f"({os.path.getsize(ENCODED_PATH) / 1e6:.1f} MB)")
print("Saved:", ENCODERS_PATH)

## Phase 8 – Train / validation / test split

- **Grouped by `orig_id`:** a real session and its anomalous copy always land in the same split. Otherwise the model would see the original in training and its near-identical twin in test, and the test score would be inflated.
- **Stratified** on "the group contains an anomaly", so the three splits have the same anomaly rate.
- 70 / 15 / 15, fixed seed. The test set is used **once**, in Phase 11; every decision (early stopping, threshold) is taken on validation.

In [ ]:
with open(os.path.join(OUT_DIR, "encoders.json")) as f:
    ENCODERS = json.load(f)
arrays = np.load(os.path.join(OUT_DIR, "sequences_encoded.npz"))

X_action, X_aisle, X_num = arrays["X_action"], arrays["X_aisle"], arrays["X_num"]
y, lengths = arrays["y"], arrays["lengths"]
session_id, orig_id, pattern = arrays["session_id"], arrays["orig_id"], arrays["pattern"]

MAX_LEN = ENCODERS["max_len"]
PAD = ENCODERS["pad_id"]
ACTION_VOCAB = ENCODERS["action_vocab"]
AISLE_OFFSET = ENCODERS["aisle_offset"]
AISLE_VOCAB_SIZE = ENCODERS["aisle_vocab_size"]
NUM_FEATURES = ENCODERS["num_features"]

print(f"{len(y):,} sessions loaded | anomaly rate {y.mean():.1%}")

In [ ]:
# One group = one real session + its anomalous copy (if any); a group is "positive" if it holds an anomaly
group_has_anomaly = pd.Series(y, index=orig_id).groupby(level=0).max()

test_frac = CONFIG["test_size"]
val_frac = CONFIG["val_size"] / (1 - test_frac)          # share of what remains after the test split
g_trainval, g_test = train_test_split(group_has_anomaly.index, test_size=test_frac,
                                      stratify=group_has_anomaly.values, random_state=SEED)
g_train, g_val = train_test_split(g_trainval, test_size=val_frac,
                                  stratify=group_has_anomaly.loc[g_trainval].values, random_state=SEED)

split_of_group = {**dict.fromkeys(g_train, "train"), **dict.fromkeys(g_val, "val"), **dict.fromkeys(g_test, "test")}
split = pd.Series(orig_id).map(split_of_group).to_numpy()
idx = {name: np.flatnonzero(split == name) for name in ["train", "val", "test"]}

# Every session is in exactly one split, and no group is shared between splits
assert sum(len(i) for i in idx.values()) == len(y)
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    assert not set(orig_id[idx[a]]) & set(orig_id[idx[b]]), f"Group leakage between {a} and {b}"

display(pd.DataFrame({name: {"sessions": len(i), "share": round(len(i) / len(y), 3),
                             "anomalies": int(y[i].sum()), "anomaly_rate": round(float(y[i].mean()), 3)}
                      for name, i in idx.items()}).T)
display(pd.crosstab(split, pattern, normalize="index").round(3))

In [ ]:
np.savez(os.path.join(OUT_DIR, "split_indices.npz"), **idx)
print("Saved: split_indices.npz")

## Phase 9 – Model

```
action ids ─► Embedding(4) ─┐
                            ├─► Concatenate ─► LSTM(64, all steps) ─► state at the last real event ─► Dropout ─► Dense(32, relu) ─► Dense(1, sigmoid)
aisle ids  ─► Embedding(8) ─┤                                          ▲
log_dt, log_price ─► Norm. ─┘                         last_step (one-hot of the last real position)
```

- **Embeddings** turn ids into small learnt vectors (an aisle id is a category, not a quantity).
- **`Normalization`** holds the mean/variance of the *training* events, so scaling is saved inside the model and identical at inference.
- **LSTM** reads the visit event by event; its state at the last real event summarises the whole visit.
- **How padding is ignored – exact by construction.** The LSTM returns its state at every position and the model keeps only the state at the last real event (`Dot` with a one-hot `last_step` input). With right padding, the padded positions come *after* that event, so they cannot influence it. A first version relied on Keras mask propagation; the padding test below failed on Kaggle (max difference 0.088), which showed the mask was not reaching the LSTM. This design does not depend on mask propagation at all, and the test now injects noise into all three inputs.
- Output: probability that the session is an anomaly (the **anomaly score**).

In [ ]:
def last_step_onehot(lengths):
    """One-hot of the last real position of each session: tells the model where the visit ends."""
    onehot = np.zeros((len(lengths), MAX_LEN), dtype="float32")
    onehot[np.arange(len(lengths)), np.asarray(lengths) - 1] = 1.0
    return onehot


def take(i):
    return {"action": X_action[i], "aisle": X_aisle[i], "numeric": X_num[i],
            "last_step": last_step_onehot(lengths[i])}, y[i]


X_all, _ = take(np.arange(len(y)))
X_train, y_train = take(idx["train"])
X_val, y_val = take(idx["val"])
X_test, y_test = take(idx["test"])

# Scaling statistics from real (non-padded) training events only
train_events = X_num[idx["train"]][X_action[idx["train"]] != PAD]
num_mean, num_var = train_events.mean(axis=0), train_events.var(axis=0)
print("Numeric mean:", num_mean.round(3), "| variance:", num_var.round(3))

In [ ]:
HP = {
    "emb_action": 4,
    "emb_aisle": 8,
    "lstm_units": 64,
    "dense_units": 32,
    "dropout": 0.3,
    "learning_rate": 1e-3,
    "batch_size": 256,
    "max_epochs": 40,
    "early_stopping_patience": 6,
}


def build_model(hp, mean, variance):
    action_in = keras.Input(shape=(MAX_LEN,), dtype="int32", name="action")
    aisle_in = keras.Input(shape=(MAX_LEN,), dtype="int32", name="aisle")
    numeric_in = keras.Input(shape=(MAX_LEN, len(NUM_FEATURES)), dtype="float32", name="numeric")
    last_in = keras.Input(shape=(MAX_LEN,), dtype="float32", name="last_step")

    a = layers.Embedding(len(ACTION_VOCAB) + 1, hp["emb_action"], name="action_emb")(action_in)
    s = layers.Embedding(AISLE_VOCAB_SIZE, hp["emb_aisle"], name="aisle_emb")(aisle_in)
    x = layers.Normalization(axis=-1, mean=mean, variance=variance, name="numeric_scaler")(numeric_in)

    h = layers.Concatenate(name="event_features")([a, s, x])
    h = layers.LSTM(hp["lstm_units"], return_sequences=True, name="lstm")(h)   # state at every position
    h = layers.Dot(axes=1, name="last_real_state")([h, last_in])                 # keep the last real event
    h = layers.Dropout(hp["dropout"], name="dropout")(h)
    h = layers.Dense(hp["dense_units"], activation="relu", name="dense")(h)
    out = layers.Dense(1, activation="sigmoid", name="anomaly_score")(h)

    model = keras.Model(inputs={"action": action_in, "aisle": aisle_in, "numeric": numeric_in, "last_step": last_in},
                        outputs=out, name="trustcart_rnn")
    model.compile(
        optimizer=keras.optimizers.Adam(hp["learning_rate"]),
        loss="binary_crossentropy",
        metrics=[keras.metrics.AUC(curve="PR", name="pr_auc"), keras.metrics.AUC(name="roc_auc"),
                 keras.metrics.Precision(name="precision"), keras.metrics.Recall(name="recall")],
    )
    return model


keras.utils.set_random_seed(SEED)
model = build_model(HP, num_mean, num_var)
model.summary()

# Padding test: random values written into the padding of every input must not change any prediction
probe = {k: v[:256].copy() for k, v in X_val.items()}
noisy = {k: v.copy() for k, v in probe.items()}
pad = noisy["action"] == PAD
noise_rng = np.random.default_rng(SEED)
noisy["action"][pad] = noise_rng.integers(1, len(ACTION_VOCAB) + 1, size=pad.sum())
noisy["aisle"][pad] = noise_rng.integers(1, AISLE_VOCAB_SIZE, size=pad.sum())
noisy["numeric"][pad] = noise_rng.normal(size=(pad.sum(), len(NUM_FEATURES))) * 5
diff = np.abs(model.predict(probe, verbose=0) - model.predict(noisy, verbose=0)).max()
assert diff < 1e-5, f"Padding leaks into the prediction (max diff {diff:.2e})"
print(f"Padding test passed: padding has no effect on the output (max diff {diff:.1e}).")

# Save / load round trip: the exported model must give the same predictions
roundtrip_path = os.path.join(OUT_DIR, "_roundtrip.keras")
model.save(roundtrip_path)
reloaded = keras.models.load_model(roundtrip_path)
diff = np.abs(model.predict(probe, verbose=0) - reloaded.predict(probe, verbose=0)).max()
os.remove(roundtrip_path)
assert diff < 1e-6, f"Reloaded model differs (max diff {diff:.2e})"
print(f"Save/load round trip passed (max diff {diff:.1e}).")

## Phase 10 – Training

- **Loss:** binary cross-entropy. **Class weights** (inverse frequency) so the 9 % of anomalies weigh as much as the 91 % of normal sessions.
- **Monitored metric: validation PR-AUC.** With rare positives, accuracy is meaningless (always answering "normal" scores 91 %); PR-AUC measures how well anomalies are ranked above normal sessions.
- **Early stopping** restores the best epoch; **ReduceLROnPlateau** halves the learning rate when progress stalls.

In [ ]:
counts = np.bincount(y_train)
class_weight = {c: len(y_train) / (2 * n) for c, n in enumerate(counts)}
print("Class weights:", {k: round(v, 2) for k, v in class_weight.items()})

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_pr_auc", mode="max",
                                  patience=HP["early_stopping_patience"], restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_pr_auc", mode="max", factor=0.5, patience=3, min_lr=1e-5),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=HP["max_epochs"],
    batch_size=HP["batch_size"],
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=2,
)
best_epoch = int(np.argmax(history.history["val_pr_auc"])) + 1
print(f"Best epoch: {best_epoch} | val PR-AUC {max(history.history['val_pr_auc']):.4f}")

In [ ]:
hist = pd.DataFrame(history.history)
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
for axis, metric in zip(ax, ["loss", "pr_auc", "roc_auc"]):
    axis.plot(hist.index + 1, hist[metric], label="train")
    axis.plot(hist.index + 1, hist[f"val_{metric}"], label="validation")
    axis.axvline(best_epoch, color="grey", linestyle="--", linewidth=1)
    axis.set_title(metric); axis.set_xlabel("epoch"); axis.legend()
plt.tight_layout(); plt.show()

## Phase 11 – Evaluation

1. **Threshold chosen on validation** (best F1), never on test.
2. **Test metrics, computed once:** ROC-AUC, PR-AUC, precision, recall, false-alarm rate, and detection rate per theft pattern.
3. **Baselines** trained on the same split:
   - *length only*: measures the residual length shortcut left by the injection (Phase 6). Close to 0.5 ROC-AUC = no usable shortcut.
   - *session aggregates* (counts, max pause, prices, aisles) in a logistic regression: shows what the sequence model adds over order-free statistics.

In [ ]:
p_val = model.predict(X_val, batch_size=1024, verbose=0).ravel()
p_test = model.predict(X_test, batch_size=1024, verbose=0).ravel()

precision_v, recall_v, thresholds_v = precision_recall_curve(y_val, p_val)
f1_v = 2 * precision_v * recall_v / np.clip(precision_v + recall_v, 1e-12, None)
best = int(np.argmax(f1_v[:-1]))                  # last point has no threshold
THRESHOLD = float(thresholds_v[best])
print(f"Threshold (best validation F1 = {f1_v[best]:.3f}): {THRESHOLD:.3f}")

In [ ]:
pred_test = (p_test >= THRESHOLD).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, pred_test, labels=[0, 1]).ravel()

TEST_METRICS = {
    "roc_auc": float(roc_auc_score(y_test, p_test)),
    "pr_auc": float(average_precision_score(y_test, p_test)),
    "precision": tp / max(tp + fp, 1),
    "recall": tp / max(tp + fn, 1),
    "false_alarm_rate": fp / max(fp + tn, 1),
    "threshold": THRESHOLD,
}
TEST_METRICS["f1"] = 2 * TEST_METRICS["precision"] * TEST_METRICS["recall"] / max(
    TEST_METRICS["precision"] + TEST_METRICS["recall"], 1e-12)

print(pd.Series(TEST_METRICS).round(4).to_string())
print(f"\nConfusion matrix (test): TN={tn}  FP={fp}  FN={fn}  TP={tp}")
print(classification_report(y_test, pred_test, target_names=["normal", "anomaly"], digits=3))

by_pattern = (pd.DataFrame({"pattern": pattern[idx["test"]], "score": p_test, "flagged": pred_test})
                .groupby("pattern").agg(sessions=("score", "size"), mean_score=("score", "mean"),
                                        flagged_rate=("flagged", "mean")).round(3))
display(by_pattern)     # flagged_rate = recall for theft patterns, false-alarm rate for "none"

In [ ]:
def session_aggregates(i):
    """Order-free statistics of each session, computed from the encoded arrays."""
    a, aisle, dt, price = X_action[i], X_aisle[i], X_num[i, :, 0], X_num[i, :, 1]
    real = a != PAD
    n = real.sum(axis=1)
    return pd.DataFrame({
        "length": n,
        "n_look": (a == ACTION_VOCAB["LOOK"]).sum(axis=1),
        "n_pick": (a == ACTION_VOCAB["PICK"]).sum(axis=1),
        "n_pay": (a == ACTION_VOCAB["PAY"]).sum(axis=1),
        "max_log_dt": np.where(real, dt, 0).max(axis=1),
        "mean_log_dt": (dt * real).sum(axis=1) / n,
        "max_log_price": np.where(real, price, 0).max(axis=1),
        "mean_log_price": (price * real).sum(axis=1) / n,
        "max_pick_log_price": np.where(a == ACTION_VOCAB["PICK"], price, 0).max(axis=1),
        "n_aisles": [len(set(row[row != PAD])) for row in aisle],
    })


agg_train, agg_test = session_aggregates(idx["train"]), session_aggregates(idx["test"])
BASELINES = {}
for name, cols in [("length only", ["length"]), ("session aggregates", list(agg_train.columns))]:
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
    clf.fit(agg_train[cols], y_train)
    p = clf.predict_proba(agg_test[cols])[:, 1]
    BASELINES[name] = {"roc_auc": float(roc_auc_score(y_test, p)),
                       "pr_auc": float(average_precision_score(y_test, p)), "scores": p}

comparison = pd.DataFrame({name: {k: v for k, v in b.items() if k != "scores"} for name, b in BASELINES.items()})
comparison["LSTM (ours)"] = {"roc_auc": TEST_METRICS["roc_auc"], "pr_auc": TEST_METRICS["pr_auc"]}
display(comparison.T.round(4))
print(f"Test anomaly rate (PR-AUC of a random model): {y_test.mean():.3f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
curves = {"LSTM (ours)": p_test, **{name: b["scores"] for name, b in BASELINES.items()}}
for name, scores in curves.items():
    fpr, tpr, _ = roc_curve(y_test, scores)
    ax[0].plot(fpr, tpr, label=f"{name} ({roc_auc_score(y_test, scores):.3f})")
    prec, rec, _ = precision_recall_curve(y_test, scores)
    ax[1].plot(rec, prec, label=f"{name} ({average_precision_score(y_test, scores):.3f})")
ax[0].plot([0, 1], [0, 1], "k--", linewidth=1)
ax[0].set(title="ROC curve (test)", xlabel="false positive rate", ylabel="true positive rate")
ax[1].axhline(y_test.mean(), color="k", linestyle="--", linewidth=1)
ax[1].set(title="Precision-recall curve (test)", xlabel="recall", ylabel="precision")
for axis in ax:
    axis.legend(fontsize=8)
plt.tight_layout(); plt.show()

## Phase 12 – Error analysis

Look at *what* the model gets wrong, not only how often: score distribution per pattern, then the most confident mistakes decoded back to readable events.

In [ ]:
test_scores = pd.DataFrame({"pattern": pattern[idx["test"]], "score": p_test})
fig, ax = plt.subplots(figsize=(9, 3.5))
for name, g in test_scores.groupby("pattern"):
    ax.hist(g["score"], bins=40, range=(0, 1), alpha=0.5, density=True, label=name)
ax.axvline(THRESHOLD, color="k", linestyle="--", linewidth=1, label="threshold")
ax.set(title="Anomaly score by pattern (test)", xlabel="score"); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
INV_ACTION = {v: k for k, v in ACTION_VOCAB.items()}


def decode(i):
    """Turn an encoded session back into readable events."""
    n = lengths[i]
    return pd.DataFrame({
        "action": [INV_ACTION[int(a)] for a in X_action[i, :n]],
        "aisle": X_aisle[i, :n] - AISLE_OFFSET,
        "dt_sec": np.expm1(X_num[i, :n, 0]).round(1),
        "price": np.expm1(X_num[i, :n, 1]).round(2),
    })


test_idx = idx["test"]
missed = test_idx[(y_test == 1) & (pred_test == 0)][np.argsort(p_test[(y_test == 1) & (pred_test == 0)])]
false_alarms = test_idx[(y_test == 0) & (pred_test == 1)][np.argsort(-p_test[(y_test == 0) & (pred_test == 1)])]
score_of = dict(zip(test_idx, p_test))

for title, rows in [("MISSED ANOMALIES (lowest scores)", missed[:2]), ("FALSE ALARMS (highest scores)", false_alarms[:2])]:
    print(f"\n========== {title} ==========")
    for i in rows:
        print(f"session {session_id[i]} | pattern {pattern[i]} | score {score_of[i]:.3f}")
        display(decode(i))

## Phase 13 – Export

Everything the rest of TrustCart needs, written to `/kaggle/working`:

| File | Used by |
|---|---|
| `rnn_model.keras` | inference |
| `rnn_meta.json` | threshold, test metrics, hyper-parameters |
| `encoders.json` | identical preprocessing at inference (Phase 7) |
| `rnn_inference.py` | `SessionScorer`: raw events → anomaly score (logistic regression, Streamlit app) |
| `rnn_scores.parquet` | RNN score of every session, with its split (training data for the logistic regression) |

The last cell re-scores test sessions **from raw events** through `rnn_inference.py` and checks it matches the notebook, so the exported scorer is proven identical to what was evaluated.

In [ ]:
MODEL_PATH = os.path.join(OUT_DIR, "rnn_model.keras")
META_PATH = os.path.join(OUT_DIR, "rnn_meta.json")
SCORES_PATH = os.path.join(OUT_DIR, "rnn_scores.parquet")

model.save(MODEL_PATH)

p_all = model.predict(X_all, batch_size=1024, verbose=0).ravel()
pd.DataFrame({"session_id": session_id, "orig_id": orig_id, "split": split, "label": y,
              "pattern": pattern, "rnn_score": p_all}).to_parquet(SCORES_PATH, index=False)

meta = {
    "threshold": THRESHOLD,
    "test_metrics": TEST_METRICS,
    "baselines_test": {name: {k: v for k, v in b.items() if k != "scores"} for name, b in BASELINES.items()},
    "hyperparameters": HP,
    "best_epoch": best_epoch,
    "split_sizes": {name: int(len(i)) for name, i in idx.items()},
    "tensorflow_version": tf.__version__,
    "keras_version": keras.__version__,
}
with open(META_PATH, "w") as f:
    json.dump(meta, f, indent=2, default=float)

for p in [MODEL_PATH, META_PATH, SCORES_PATH]:
    print("Saved:", p, f"({os.path.getsize(p) / 1e6:.2f} MB)")

In [ ]:
INFERENCE_CODE = '''"""Score a TrustCart customer session with the trained RNN (exported by rnn.ipynb, Phase 13)."""
import json

import numpy as np
from tensorflow import keras


class SessionScorer:
    """Raw session events -> anomaly score, with exactly the preprocessing used in training."""

    def __init__(self, model_path, encoders_path, meta_path):
        self.model = keras.models.load_model(model_path)
        with open(encoders_path) as f:
            self.enc = json.load(f)
        with open(meta_path) as f:
            self.threshold = float(json.load(f)["threshold"])

    def encode(self, actions, aisles, dt_sec, price):
        """actions: LOOK/PICK/PAY, aisles: store aisle (0 = unknown), dt_sec: seconds since previous action."""
        max_len, pad, n_aisles = self.enc["max_len"], self.enc["pad_id"], self.enc["n_aisles"]
        actions = list(actions)[-max_len:]
        if not actions:
            raise ValueError("Cannot score an empty session")
        aisles = np.asarray(aisles, dtype="int64")[-max_len:]
        dt_sec = np.clip(np.asarray(dt_sec, dtype="float32")[-max_len:], 0, None).copy()
        price = np.clip(np.asarray(price, dtype="float32")[-max_len:], 0, None)
        dt_sec[0] = 0.0
        aisles = np.where((aisles >= 0) & (aisles < n_aisles), aisles, 0)

        n = len(actions)
        action_ids = np.full((1, max_len), pad, dtype="int32")
        aisle_ids = np.full((1, max_len), pad, dtype="int32")
        numeric = np.zeros((1, max_len, len(self.enc["num_features"])), dtype="float32")
        action_ids[0, :n] = [self.enc["action_vocab"][a] for a in actions]
        aisle_ids[0, :n] = aisles + self.enc["aisle_offset"]
        numeric[0, :n, 0] = np.log1p(dt_sec)
        numeric[0, :n, 1] = np.log1p(price)
        last_step = np.zeros((1, max_len), dtype="float32")
        last_step[0, n - 1] = 1.0
        return {"action": action_ids, "aisle": aisle_ids, "numeric": numeric, "last_step": last_step}

    def score(self, actions, aisles, dt_sec, price):
        inputs = self.encode(actions, aisles, dt_sec, price)
        return float(np.asarray(self.model(inputs, training=False)).ravel()[0])

    def is_anomaly(self, actions, aisles, dt_sec, price):
        return self.score(actions, aisles, dt_sec, price) >= self.threshold
'''

with open(os.path.join(OUT_DIR, "rnn_inference.py"), "w") as f:
    f.write(INFERENCE_CODE)
print("Saved:", os.path.join(OUT_DIR, "rnn_inference.py"))

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location("rnn_inference", os.path.join(OUT_DIR, "rnn_inference.py"))
rnn_inference = importlib.util.module_from_spec(spec)
spec.loader.exec_module(rnn_inference)

scorer = rnn_inference.SessionScorer(MODEL_PATH, os.path.join(OUT_DIR, "encoders.json"), META_PATH)

# Re-score test sessions from the raw events of Phase 6 and compare with the notebook scores
raw = pd.read_parquet(os.path.join(OUT_DIR, "sessions_labeled.parquet")).sort_values(["session_id", "step"])
rows_of = raw.groupby("session_id").indices
check = np.random.default_rng(SEED).choice(idx["test"], size=200, replace=False)

diffs = []
for i in check:
    ev = raw.iloc[rows_of[session_id[i]]]
    diffs.append(abs(scorer.score(ev["action"], ev["aisle"], ev["dt_sec"], ev["price"]) - p_all[i]))
assert max(diffs) < 1e-3, f"Exported scorer differs from the notebook (max diff {max(diffs):.2e})"
print(f"Exported scorer matches the notebook on 200 test sessions (max diff {max(diffs):.1e}).")

## Phase 14 – Robustness

Four questions an examiner will ask about the numbers above:

| Experiment | Question |
|---|---|
| **A. Several seeds** | Is the test score stable, or one lucky run? → metrics as mean ± std over 4 trainings |
| **B. Leave one pattern out** | Does the model detect a theft behaviour it has *never seen*? → train without one pattern, test on it |
| **C. False alarms** | Who are the honest customers flagged with a very high score? → do they naturally contain a theft cue? |
| **D. Calibration** | Does a score of 0.9 mean a 90 % risk? → Platt scaling fitted on validation, reliability diagram |

Every model in this phase uses the same architecture, hyper-parameters, early stopping and validation-only threshold as the main model.

In [ ]:
def fit_model(train_i, val_i, seed, verbose=0):
    """Train a fresh model on train_i with early stopping on val_i; scaling stats from train_i only."""
    keras.utils.set_random_seed(seed)
    events = X_num[train_i][X_action[train_i] != PAD]
    m = build_model(HP, events.mean(axis=0), events.var(axis=0))
    (X_tr, y_tr), (X_va, y_va) = take(train_i), take(val_i)
    cw = {c: len(y_tr) / (2 * n) for c, n in enumerate(np.bincount(y_tr, minlength=2))}
    m.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=HP["max_epochs"], batch_size=HP["batch_size"],
          class_weight=cw, verbose=verbose, callbacks=[
              keras.callbacks.EarlyStopping(monitor="val_pr_auc", mode="max",
                                            patience=HP["early_stopping_patience"], restore_best_weights=True),
              keras.callbacks.ReduceLROnPlateau(monitor="val_pr_auc", mode="max", factor=0.5, patience=3, min_lr=1e-5),
          ])
    return m


def predict(m, i):
    return m.predict(take(i)[0], batch_size=1024, verbose=0).ravel()


def best_f1_threshold(y_true, scores):
    prec, rec, thr = precision_recall_curve(y_true, scores)
    f1 = 2 * prec * rec / np.clip(prec + rec, 1e-12, None)
    return float(thr[int(np.argmax(f1[:-1]))])


def evaluate(y_true, scores, threshold):
    pred = scores >= threshold
    tp, fp = int((pred & (y_true == 1)).sum()), int((pred & (y_true == 0)).sum())
    fn, tn = int((~pred & (y_true == 1)).sum()), int((~pred & (y_true == 0)).sum())
    return {"roc_auc": float(roc_auc_score(y_true, scores)),
            "pr_auc": float(average_precision_score(y_true, scores)),
            "precision": tp / max(tp + fp, 1), "recall": tp / max(tp + fn, 1),
            "false_alarm_rate": fp / max(fp + tn, 1)}

In [ ]:
# A. Same data and split, different random seeds (weight initialisation, batch order, dropout)
METRIC_COLS = ["roc_auc", "pr_auc", "precision", "recall", "false_alarm_rate"]
seed_rows = [{"seed": SEED, **{k: TEST_METRICS[k] for k in METRIC_COLS}}]
for seed in [SEED + 1, SEED + 2, SEED + 3]:
    m = fit_model(idx["train"], idx["val"], seed)
    seed_rows.append({"seed": seed, **evaluate(y_test, predict(m, idx["test"]),
                                               best_f1_threshold(y_val, predict(m, idx["val"])))})

seeds = pd.DataFrame(seed_rows).set_index("seed")
SEED_SUMMARY = seeds.agg(["mean", "std"])
display(seeds.round(4))
display(SEED_SUMMARY.round(4))

In [ ]:
# B. Leave one pattern out: that pattern is removed from train and validation, then it is the only
#    kind of theft in the test set. Normal test sessions are unchanged.
lopo_rows = []
for held in ["conceal", "alibi", "spree"]:
    tr = idx["train"][pattern[idx["train"]] != held]
    va = idx["val"][pattern[idx["val"]] != held]
    te = idx["test"][np.isin(pattern[idx["test"]], ["none", held])]
    m = fit_model(tr, va, SEED)
    result = evaluate(y[te], predict(m, te), best_f1_threshold(y[va], predict(m, va)))
    lopo_rows.append({"held_out_pattern": held,
                      "recall_when_seen": float(by_pattern.loc[held, "flagged_rate"]), **result})

lopo = pd.DataFrame(lopo_rows).set_index("held_out_pattern")
display(lopo.round(3))      # recall = detection of a pattern never seen in training

In [ ]:
# C. False alarms: do honest customers flagged by the model naturally contain a theft cue?
with open(os.path.join(OUT_DIR, "injection_config.json")) as f:
    INJ_CFG = json.load(f)


def theft_cues(i):
    """Whether a session contains, on its own, the cue each injected pattern is built on."""
    n = lengths[i]
    a, dt, price = X_action[i, :n], np.expm1(X_num[i, :n, 0]), np.expm1(X_num[i, :n, 1])
    pick, pay = a == ACTION_VOCAB["PICK"], a == ACTION_VOCAB["PAY"]
    expensive_pick = pick & (price > INJ_CFG["price_expensive"])
    fast_pick = pick[1:] & pick[:-1] & (dt[1:] <= INJ_CFG["dt_fast"])          # pick quickly after a pick
    return {
        "conceal_cue": bool(np.any(expensive_pick[:-1] & (dt[1:] >= INJ_CFG["dt_long_lo"]))),
        "spree_cue": bool(np.any(fast_pick[1:] & fast_pick[:-1])),            # 3 picks, 2 fast gaps
        "alibi_cue": bool(expensive_pick.any() and pay.any() and price[pay].max() <= INJ_CFG["price_cheap"]),
    }


normal_test = idx["test"][y_test == 0]
flagged = p_test[y_test == 0] >= THRESHOLD
cues = pd.DataFrame([theft_cues(i) for i in normal_test])
cues["any_cue"] = cues.any(axis=1)
cues["group"] = np.where(flagged, "false alarm", "correctly accepted")
FALSE_ALARM_CUES = cues.groupby("group").mean()
FALSE_ALARM_CUES["sessions"] = cues.groupby("group").size()
display(FALSE_ALARM_CUES.round(3))

score_of = dict(zip(idx["test"], p_test))
for i in normal_test[flagged][np.argsort(-p_test[y_test == 0][flagged])][:3]:
    print(f"\nFalse alarm | session {session_id[i]} | score {score_of[i]:.3f} | cues {theft_cues(i)}")
    display(decode(i))

In [ ]:
# D. Calibration: class weights push scores towards 0 and 1. Platt scaling (a logistic regression on
#    the logit of the score) is fitted on validation; it is monotonic, so it changes no decision.
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


platt = LogisticRegression(C=1e6).fit(logit(p_val)[:, None], y_val)
CAL_A, CAL_B = float(platt.coef_[0, 0]), float(platt.intercept_[0])


def calibrate(p):
    return 1 / (1 + np.exp(-(CAL_A * logit(p) + CAL_B)))


p_test_cal = calibrate(p_test)
THRESHOLD_CAL = float(calibrate(np.array([THRESHOLD]))[0])
assert ((p_test_cal >= THRESHOLD_CAL) == (p_test >= THRESHOLD)).all(), "Calibration changed a decision"


def expected_calibration_error(y_true, p, bins=10):
    bin_of = np.minimum((p * bins).astype(int), bins - 1)
    return float(sum(abs(p[bin_of == b].mean() - y_true[bin_of == b].mean()) * (bin_of == b).mean()
                     for b in range(bins) if (bin_of == b).any()))


CALIBRATION = {
    "method": "platt", "a": CAL_A, "b": CAL_B, "threshold_calibrated": THRESHOLD_CAL,
    "brier_before": float(brier_score_loss(y_test, p_test)), "brier_after": float(brier_score_loss(y_test, p_test_cal)),
    "ece_before": expected_calibration_error(y_test, p_test), "ece_after": expected_calibration_error(y_test, p_test_cal),
}
print(pd.Series({k: v for k, v in CALIBRATION.items() if k != "method"}).round(4).to_string())

fig, ax = plt.subplots(figsize=(5, 5))
for name, p in [("raw score", p_test), ("calibrated", p_test_cal)]:
    frac_pos, mean_pred = calibration_curve(y_test, p, n_bins=10, strategy="quantile")
    ax.plot(mean_pred, frac_pos, marker="o", label=name)
ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="perfect")
ax.set(title="Reliability diagram (test)", xlabel="predicted risk", ylabel="observed theft rate")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Export v2: calibrated scores for the logistic regression and the app, plus the robustness results
p_all_cal = calibrate(p_all)
pd.DataFrame({"session_id": session_id, "orig_id": orig_id, "split": split, "label": y, "pattern": pattern,
              "rnn_score": p_all_cal, "rnn_score_raw": p_all}).to_parquet(SCORES_PATH, index=False)

with open(META_PATH) as f:
    meta = json.load(f)
meta["calibration"] = CALIBRATION
meta["robustness"] = {
    "seeds": {"runs": seeds.round(4).reset_index().to_dict("records"),
              "mean": SEED_SUMMARY.loc["mean"].round(4).to_dict(), "std": SEED_SUMMARY.loc["std"].round(4).to_dict()},
    "leave_one_pattern_out": lopo.round(4).reset_index().to_dict("records"),
    "false_alarm_cues": FALSE_ALARM_CUES.round(4).reset_index().to_dict("records"),
}
with open(META_PATH, "w") as f:
    json.dump(meta, f, indent=2, default=float)

INFERENCE_CODE_V2 = INFERENCE_CODE.replace(
    '''        with open(meta_path) as f:
            self.threshold = float(json.load(f)["threshold"])''',
    '''        with open(meta_path) as f:
            meta = json.load(f)
        calibration = meta["calibration"]
        self.cal_a, self.cal_b = calibration["a"], calibration["b"]
        self.threshold = float(calibration["threshold_calibrated"])     # threshold on the calibrated score'''
).replace(
    '''    def score(self, actions, aisles, dt_sec, price):
        inputs = self.encode(actions, aisles, dt_sec, price)
        return float(np.asarray(self.model(inputs, training=False)).ravel()[0])''',
    '''    def raw_score(self, actions, aisles, dt_sec, price):
        """Uncalibrated model output."""
        inputs = self.encode(actions, aisles, dt_sec, price)
        return float(np.asarray(self.model(inputs, training=False)).ravel()[0])

    def score(self, actions, aisles, dt_sec, price):
        """Calibrated theft risk in [0, 1] (Platt scaling fitted on validation)."""
        p = min(max(self.raw_score(actions, aisles, dt_sec, price), 1e-6), 1 - 1e-6)
        return float(1 / (1 + np.exp(-(self.cal_a * np.log(p / (1 - p)) + self.cal_b))))'''
)
assert "raw_score" in INFERENCE_CODE_V2 and "threshold_calibrated" in INFERENCE_CODE_V2, "Inference patch failed"
with open(os.path.join(OUT_DIR, "rnn_inference.py"), "w") as f:
    f.write(INFERENCE_CODE_V2)

# The exported calibrated scorer must reproduce the notebook on raw events
spec = importlib.util.spec_from_file_location("rnn_inference_v2", os.path.join(OUT_DIR, "rnn_inference.py"))
rnn_inference = importlib.util.module_from_spec(spec)
spec.loader.exec_module(rnn_inference)
scorer = rnn_inference.SessionScorer(MODEL_PATH, os.path.join(OUT_DIR, "encoders.json"), META_PATH)

diffs = []
for i in check:
    ev = raw.iloc[rows_of[session_id[i]]]
    diffs.append(abs(scorer.score(ev["action"], ev["aisle"], ev["dt_sec"], ev["price"]) - p_all_cal[i]))
assert max(diffs) < 1e-4, f"Calibrated scorer differs from the notebook (max diff {max(diffs):.2e})"
print(f"Calibrated scorer matches the notebook on 200 test sessions (max diff {max(diffs):.1e}).")
print("Updated:", SCORES_PATH, "|", META_PATH, "|", os.path.join(OUT_DIR, "rnn_inference.py"))